In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler

# Load dataset (Replace with your exact CSV filename)
df = pd.read_csv('data_ecommerce_customer_churn.csv')
print('Dataset Shape:', df.shape)
df.head()

In [ ]:
# Check missing values
print('Missing Values:\n', df.isnull().sum())

# Fill missing numerical values with median, or drop if necessary
df.fillna(df.median(numeric_only=True), inplace=True)

In [ ]:
# Select a key numerical column to treat outliers (e.g., Balance, Tenure, or Amount)
num_col = 'Balance'  # Replace with a numeric column from your dataset

Q1 = df[num_col].quantile(0.25)
Q3 = df[num_col].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Filter outliers
df = df[(df[num_col] >= lower_bound) & (df[num_col] <= upper_bound)]

# Visualize with a Box Plot
plt.figure(figsize=(6, 3))
sns.boxplot(x=df[num_col])
plt.title(f'Box Plot of {num_col} after Outlier Treatment')
plt.show()

In [ ]:
# Statistical Summary
print(df.describe())

# Correlation Heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(df.corr(numeric_only=True), annot=False, cmap='coolwarm')
plt.title('Feature Correlation Heatmap')
plt.show()

In [ ]:
# Drop irrelevant identifier columns (like Customer ID, Surname, etc. if present)
df = df.drop(
    columns=[
        col
        for col in ['CustomerID', 'RowNumber', 'Customerld', 'id']
        if col in df.columns
    ],
    errors='ignore',
)

# Convert categorical text columns into numerical values
df = pd.get_dummies(df, drop_first=True)

# Separate Features (X) and Target (y) - Change 'Exited' or 'Churn' to your target column name
target_column = 'Exited'  # or 'Churn' depending on your dataset
X = df.drop(columns=[target_column])
y = df[target_column]

# Train-Test Split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Train Model
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_scaled, y_train)

# Predictions
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

# Evaluation Metrics
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred))
print('\nClassification Report:\n', classification_report(y_test, y_pred))
print('AUC-ROC Score:', roc_auc_score(y_test, y_prob))